Реляционная БД - набор таблиц с возможностью связываться между собой по ключу.

SQLite - одна из библиотек, реализующих API с СУБД по принципу 'один пишет - многие читают'.

In [1]:
import sqlite3 as sq

In [ ]:
with sq.connect('saper.db') as con:
    cur = con.cursor()

    # cur.execute('DROP TABLE IF EXISTS users')
    cur.execute("""CREATE TABLE IF NOT EXISTS users (
        user_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        sex INTEGER NOT NULL DEFAULT 1,
        old INTEGER,
        score INTEGER
    )
                """)

INSERT - добавление записи

INSERT INTO `table_name` (`column_name_1`, `column_name_2`, ...) VALUES (`value_1`, `value_2`)

INSERT INTO VALUES (`value_1`, `value_2`)


SELECT - выборка данных из таблиц

SELECT `column_name_1`, `column_name_2`, ... FROM `table_name` WHERE `condition (стандартные операторы, IN, BETWEEEN)`

SELECT * FROM `table_name` ORDER BY `col_name` [`DESC`] LIMIT `max` [OFFSET `offset`]

In [6]:
query = '''
    SELECT *
    FROM users
    WHERE score > 100
    ORDER BY score DESC
    LIMIT 5
'''

with sq.connect('saper.db') as con:
    cur = con.cursor()
    cur.execute(query)
    # result = cur.fetchall()
    # print(result)
    for res in cur:
        print(res)

(5, 'Сергей', 1, 33, 2000)
(1, 'Михаил', 1, 19, 1000)
(8, 'Юля', 2, 23, 700)
(3, 'Николай', 1, 22, 500)
(7, 'Елена', 2, 17, 500)


UPDATE - изменение данных в записях

UPDATE `table_name` SET `column_name_1 = new_value_1`, `column_name_2 = new_value_2` WHERE `condition`

UPDATE `table_name` SET `column_name = column_name + value`


DELETE - удаление записей из таблицы

DELETE FROM `table_name` WHERE `condition`

SELECT `agg_func(column)` as `alias` FROM `table_name` WHERE `condition`

SELECT  DISTINCT `column`  FROM `table_name` - только уникальные значения

SELECT `agg_func(` DISTINCT `column)`  FROM `table_name` - агрегация только по уникальным значениям


SELECT user_id, sum(user_id) as sum  
FROM games  
WHERE score > 300  
GROUP BY user_id  
ORDER BY sum DESC  
LIMIT 1

JOIN `таблица` ON `условие связывания`

SELECT name, sex, games.score  
FROM games  
JOIN users  
ON games.user_id == users.rowid

INNER  
LEFT  
RIGHT  
OUTER  
FULL  

UNION - оставляет только уникальные записи

SELECT score, `from` FROM tab1  
UNION SELECT val, type FROM tab2  

SELECT score, 'table 1' as tb1 FROM tab1  
UNION SELECT val, 'table 2' FROM tab2  
ORDER BY score DESC  

SELECT name, subject, mark  
FROM marks  
JOIN students  
ON students.rowid == marks.id  
WHERE mark > (SELECT mark  
              FROM marks  
              WHERE id == 2 AND subject LIKE 'Си')
              AND subject LIKE 'Си'


In [11]:
cars = [
    ('Audi', 52642),
    ('Mercedes', 57127),
    ('Skoda', 9000),
    ('Volvo', 29000),
    ('Bentley', 35000)
]

with sq.connect('cars.db') as con:
    cur = con.cursor()

    cur.execute("DROP TABLE IF EXISTS cars")
    cur.execute("""CREATE TABLE IF NOT EXISTS cars (
        car_id INTEGER PRIMARY KEY AUTOINCREMENT,
        model TEXT,
        price INTEGER
    )
                """)
    
    cur.execute("INSERT INTO cars VALUES(1, 'Audi', 52642)")
    cur.execute("INSERT INTO cars VALUES(2, 'Mercedes', 57127)")
    
    # for car in cars:
    #     cur.execute("INSERT INTO cars VALUES(NULL, ?, ?)", car) 
    cur.executemany("INSERT INTO cars VALUES(NULL, ?, ?)", cars)
    cur.execute("UPDATE cars SET price = :Price WHERE model LIKE 'A%'", {'Price': 0})
    cur.executescript("""DELETE FROM cars WHERE model LIKE 'A%';
                         UPDATE cars SET price = price + 1000
    """)
    
    # при выходе из менеджера контекста выполняются
    # con.commit()
    # con.close()

In [21]:
con = None
try:
    con = sq.connect('cars.db')
    cur = con.cursor()
    
    cur.executescript("""
        DROP TABLE IF EXISTS cars;
        CREATE TABLE IF NOT EXISTS cars (
        car_id INTEGER PRIMARY KEY AUTOINCREMENT,
        model TEXT,
        price INTEGER
    );
        BEGIN;
        INSERT INTO cars VALUES(NULL, 'Audi', 52642);
        INSERT INTO cars VALUES(NULL, 'Mercedes', 57127);
        INSERT INTO cars VALUES(NULL, 'Skoda', 9000);
        INSERT INTO cars VALUES(NULL, 'Volvo', 29000);
        INSERT INTO cars VALUES(NULL, 'Bentley', 35000);
        UPDATE cars SET price = price + 1000
    """)
    
    con.commit()
    
except sq.Error as e:
    if con: con.rollback() # откатывает БД к состоянию BEGIN
    print('Ошибка вполнения запроса')
    print(e)
finally:
    if con: con.close()

In [25]:
with sq.connect('cars.db') as con:
    cur = con.cursor()

    cur.executescript("""
        CREATE TABLE IF NOT EXISTS cars (
            car_id INTEGER PRIMARY KEY AUTOINCREMENT,
            model TEXT,
            price INTEGER);
        
        CREATE TABLE IF NOT EXISTS cust (
            name TEXT,
            tr_in INTEGER,
            buy INTEGER);
    """)
    
    cur.execute("INSERT INTO cars VALUES(NULL, 'Запорожец', 1000)")
    last_row_id = cur.lastrowid
    buy_car_id = 2
    cur.execute("INSERT INTO cust VALUES ('Фёдор', ?, ?)", (last_row_id, buy_car_id))

fetchall() - возвращает число записей в виде упорядоченного списка;  
fetchmany(size) - возвращает число записей не более size;  
fetchone() - возвращает первую запись. 

In [ ]:
cars = [
    ('Audi', 52642),
    ('Mercedes', 57127),
    ('Skoda', 9000),
    ('Volvo', 29000),
    ('Bentley', 35000)
]

with sq.connect('cars.db') as con:
    con.row_factory = sq.Row # чтобы запрос вместо кортежа возвращал словарь
    cur = con.cursor()

    cur.execute("DROP TABLE IF EXISTS cars")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS cars (
            car_id INTEGER PRIMARY KEY AUTOINCREMENT,
            model TEXT,
            price INTEGER
        )
    """) 
    cur.executemany("INSERT INTO cars VALUES(NULL, ?, ?)", cars)
    cur.execute("SELECT model, price FROM cars")
    
    # rows = cur.fetchall()
    # print(rows)
    
    for row in cur:
        k1, k2 = row.keys()
        print(row[k1], row[k2])


Audi 52642
Mercedes 57127
Skoda 9000
Volvo 29000
Bentley 35000


In [ ]:
def readAva(n):
    try:
        with open(f'avas/{n}.png', 'rb') as f:
            return f.read()
    except IOError as e:
        print(e)
        return False
    
def writeAva(name, data):
    try:
        with open(name, 'wb') as f:
            f.write(data)
    except IOError as e:
        print(e)
        return False
    
    return True

with sq.connect('cars.db') as con:
    con.row_factory = sq.Row # чтобы запрос вместо кортежа возвращал словарь
    cur = con.cursor()

    cur.execute("DROP TABLE IF EXISTS users")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            name TEXT,
            ava BLOB,
            score INTEGER
        )
    """)
    
    img = readAva(1)
    if img:
        binary = sq.Binary(img)
        cur.execute("INSERT INTO users VALUES ('Николай', ?, 1000)", (binary, ))
        
    cur.execute("SELECT ava FROM users LIMIT 1")
    img = cur.fetchone()['ava']
    writeAva('out.png', img)

iterdump() - возвращает итератор для sql запросов, чтобы воспроизвести БД

In [ ]:

with sq.connect('cars.db') as con:
    cur = con.cursor()
    
    with open('sql_dump.sql', 'w') as f:
        for sql in con.iterdump():
            f.write(sql)

Создаём БД в памяти, а не на диске

In [ ]:
con = sq.connect(':memory:')
with con:
    pass